# Notebook 02 — IsolationForest 이상탐지 모델 학습 및 평가

**목표:** IsolationForest를 학습하고 contamination 하이퍼파라미터를 튜닝하여 최적 모델을 저장한다.

**핵심 알고리즘: IsolationForest**
- 비지도 학습 — 이상 레이블 없이 정상 데이터만으로 학습 가능
- 이상 데이터는 랜덤 트리에서 더 적은 분기만으로 고립됨 → 짧은 경로 = 이상
- `anomaly_score`: 낮을수록 이상 가능성 높음 (0 기준, 음수일수록 이상)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import pandas as pd
import joblib
import os
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_recall_curve, roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
os.makedirs('../models', exist_ok=True)
print('라이브러리 로드 완료')

## 1. 데이터 로드 및 분할

In [ ]:
df = pd.read_csv('../data/raw/sensor_data.csv')
features = ['temperature', 'vibration', 'current']

X = df[features].values
y = df['label'].values  # 0=정상, 1=이상 (평가용 — 학습에는 사용 안 함)

# 정상 데이터만 학습용으로 사용 (비지도 학습의 핵심)
X_normal = X[y == 0]

print(f'전체 데이터: {len(X):,}건')
print(f'학습용 (정상만): {len(X_normal):,}건')
print(f'평가용 (전체): {len(X):,}건')

## 2. 데이터 스케일링

In [ ]:
# StandardScaler: 피처별 분산이 다르므로 정규화 필수
# 온도(70±5) vs 진동(0.5±0.1) → 스케일 차이가 크면 모델이 편향될 수 있음
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)
X_all_scaled    = scaler.transform(X)

print('스케일링 전 평균:', np.round(X_normal.mean(axis=0), 3))
print('스케일링 후 평균:', np.round(X_normal_scaled.mean(axis=0), 3))
print('스케일링 후 표준편차:', np.round(X_normal_scaled.std(axis=0), 3))

## 3. Contamination 튜닝

`contamination`: 전체 데이터 중 이상이 차지하는 비율 추정값  
→ 실제 비율은 약 4.8% (500/10500) — 여러 값을 시험해 최적값 탐색

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

contaminations = [0.01, 0.03, 0.05, 0.07, 0.10]
results = []

for c in contaminations:
    model = IsolationForest(contamination=c, n_estimators=100,
                            random_state=SEED, n_jobs=-1)
    model.fit(X_normal_scaled)
    preds = model.predict(X_all_scaled)  # 1=정상, -1=이상
    y_pred = (preds == -1).astype(int)   # 이상=1로 변환

    results.append({
        'contamination': c,
        'precision': round(precision_score(y, y_pred), 3),
        'recall':    round(recall_score(y, y_pred), 3),
        'f1':        round(f1_score(y, y_pred), 3),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
best = results_df.loc[results_df['f1'].idxmax()]
print(f"\n최적 contamination: {best['contamination']} (F1={best['f1']})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results_df['contamination'], results_df['precision'], 'o-', label='Precision')
ax.plot(results_df['contamination'], results_df['recall'],    's-', label='Recall')
ax.plot(results_df['contamination'], results_df['f1'],        '^-', label='F1', linewidth=2)
ax.axvline(best['contamination'], color='red', linestyle='--', alpha=0.5, label=f"최적값 ({best['contamination']})")
ax.set_xlabel('Contamination')
ax.set_ylabel('Score')
ax.set_title('Contamination 튜닝 결과')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/tuning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 최적 모델 학습 및 저장

In [ ]:
BEST_CONTAMINATION = float(best['contamination'])

final_model = IsolationForest(
    contamination=BEST_CONTAMINATION,
    n_estimators=200,  # 안정성 위해 100→200으로 증가
    random_state=SEED,
    n_jobs=-1,
)
final_model.fit(X_normal_scaled)

joblib.dump(final_model, '../models/isolation_forest.joblib')
joblib.dump(scaler,      '../models/scaler.joblib')
print('모델 저장 완료: models/isolation_forest.joblib')
print('스케일러 저장 완료: models/scaler.joblib')

## 5. 최종 성능 평가

In [ ]:
preds_final = final_model.predict(X_all_scaled)
y_pred_final = (preds_final == -1).astype(int)
scores = final_model.score_samples(X_all_scaled)

print('=== 분류 리포트 ===')
print(classification_report(y, y_pred_final, target_names=['정상', '이상']))
print(f'ROC-AUC: {roc_auc_score(y, -scores):.3f}')

In [ ]:
# 혼동 행렬
cm = confusion_matrix(y, y_pred_final)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['정상 예측', '이상 예측'])
ax.set_yticklabels(['정상 실제', '이상 실제'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                fontsize=16, color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title('혼동 행렬 (Confusion Matrix)')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 이상 점수 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 점수 분포
ax = axes[0]
ax.hist(scores[y==0], bins=60, alpha=0.6, color='#3B82F6', label='정상', density=True)
ax.hist(scores[y==1], bins=40, alpha=0.6, color='#EF4444', label='이상', density=True)
threshold = final_model.offset_
ax.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'임계값 ({threshold:.3f})')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('밀도')
ax.set_title('이상 점수 분포')
ax.legend()

# Precision-Recall 곡선
ax2 = axes[1]
precision, recall, _ = precision_recall_curve(y, -scores)
ax2.plot(recall, precision, color='#8B5CF6', linewidth=2)
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall 곡선')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1]); ax2.set_ylim([0, 1])

plt.suptitle('모델 성능 시각화', fontsize=14)
plt.tight_layout()
plt.savefig('../data/model_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 결정 경계 시각화 (2D 슬라이스)

In [ ]:
# 온도 vs 전류 평면에서 결정 경계 시각화 (진동은 평균값 고정)
xx, yy = np.meshgrid(
    np.linspace(55, 115, 200),
    np.linspace(8, 40, 200)
)
vib_mean = X_normal[:, 1].mean()
grid = np.c_[xx.ravel(), np.full(xx.ravel().shape, vib_mean), yy.ravel()]
grid_scaled = scaler.transform(grid)
Z = final_model.score_samples(grid_scaled).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(xx, yy, Z, levels=20, cmap='RdYlBu', alpha=0.7)
plt.colorbar(cf, label='Anomaly Score')
ax.contour(xx, yy, Z, levels=[final_model.offset_], colors='black', linewidths=2)

for grp, color, label in [(0, '#1D4ED8', '정상'), (1, '#DC2626', '이상')]:
    sub = df[df.label == grp]
    ax.scatter(sub['temperature'], sub['current'], c=color, s=6, alpha=0.4, label=label)

ax.set_xlabel('온도 (°C)')
ax.set_ylabel('전류 (A)')
ax.set_title('IsolationForest 결정 경계 (온도 vs 전류, 진동 고정)')
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig('../data/decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()
print('검정 실선: 이상/정상 결정 경계')

## 8. 모델 요약

| 항목 | 값 |
|------|----|
| 알고리즘 | IsolationForest |
| 학습 방식 | 비지도 (정상 데이터만 사용) |
| n_estimators | 200 |
| contamination | 최적값 자동 선택 |
| 추론 속도 | < 5ms / sample |

**저장 경로:**
- `models/isolation_forest.joblib` — 이상탐지 모델
- `models/scaler.joblib` — 정규화 스케일러

→ FastAPI 서버 재시작 시 자동으로 이 모델을 로드하여 서빙